<a href="https://colab.research.google.com/github/codeair1/Food-Delivery-Analysis-and-Prediction/blob/main/SI_MiniProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d gauravmalik26/food-delivery-dataset
!unzip -o food-delivery-dataset.zip


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/gauravmalik26/food-delivery-dataset
License(s): other
100% 1.95M/1.95M [00:00<00:00, 169MB/s]

Archive:  food-delivery-dataset.zip
  inflating: Sample_Submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [ ]:
!pip install -q streamlit


In [ ]:
%%writefile backend.py
import io, os, glob, base64, json, sqlite3, warnings
from contextlib import closing

import numpy as np, pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.diagnostic import het_breuschpagan
from matplotlib.figure import Figure
from flask import Flask, request, jsonify

warnings.filterwarnings("ignore")

DB_PATH = "analyzer_history.db"
DATA_PATTERNS = ["/content/*.csv", "/content/*/*.csv"]
TARGET_PREFIX = "time_taken"
LEVEL_ORDER = {"traffic": ["Low", "Medium", "High", "Jam"]}
UNLOGGED = {"levels", "export", "history"}
STATE = {"clean": None, "sla": 35.0, "model": None, "formula": None}

def db_exec(query, args=()):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        rows = conn.execute(query, args).fetchall()
        conn.commit()
        return rows

def db_log(endpoint, params, note):
    db_exec("INSERT INTO analyses(ts, endpoint, params, result) VALUES (datetime('now'), ?, ?, ?)",
            (endpoint, json.dumps(params)[:300], note[:200]))

db_exec("""CREATE TABLE IF NOT EXISTS analyses(
    id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, endpoint TEXT, params TEXT, result TEXT)""")


Writing backend.py


In [ ]:
%%writefile -a backend.py
def find_dataset():
    candidates = sorted((f for pattern in DATA_PATTERNS for f in glob.glob(pattern)),
                        key=lambda f: ("train" not in os.path.basename(f).lower(), f))
    for path in candidates:
        try:
            head = pd.read_csv(path, nrows=5)
        except Exception:
            continue
        if any(str(c).strip().lower().startswith(TARGET_PREFIX) for c in head.columns):
            return path
    raise FileNotFoundError("Kaggle train.csv not found in /content. Run the Kaggle download cell at the top of the notebook first.")

def haversine_km(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = (np.sin((lat2 - lat1) * p / 2) ** 2
         + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2)
    return 2 * 6371 * np.arcsin(np.sqrt(a))

def standardize(raw):
    df = raw.copy()
    df.columns = df.columns.astype(str).str.strip()
    lookup = {c.lower(): c for c in df.columns}

    def column(*names):
        for name in names:
            if name.lower() in lookup:
                return df[lookup[name.lower()]]
        return pd.Series(np.nan, index=df.index)

    def numeric(*names):
        return pd.to_numeric(column(*names), errors="coerce")

    def category(series):
        s = series.astype(str).str.strip()
        return s.mask(s.str.lower().isin(["nan", "none", ""]), np.nan)

    target = next((c for c in df.columns if c.lower().startswith(TARGET_PREFIX)), None)
    if target is None:
        raise ValueError("dataset has no Time_taken column")

    out = pd.DataFrame(index=df.index)
    out["order_id"] = np.arange(1, len(df) + 1)
    out["delivery_time_min"] = pd.to_numeric(df[target].astype(str).str.extract(r"(\d+)")[0], errors="coerce")

    lat1, lon1 = numeric("Restaurant_latitude").abs(), numeric("Restaurant_longitude").abs()
    lat2, lon2 = numeric("Delivery_location_latitude").abs(), numeric("Delivery_location_longitude").abs()
    distance = haversine_km(lat1, lon1, lat2, lon2)
    out["distance_km"] = distance.where((lat1 > 0) & (lon1 > 0) & (distance < 50)).round(2)

    ordered = pd.to_timedelta(column("Time_Orderd", "Time_Ordered").astype(str).str.strip(), errors="coerce")
    picked = pd.to_timedelta(column("Time_Order_picked").astype(str).str.strip(), errors="coerce")
    out["prep_time_min"] = (((picked - ordered).dt.total_seconds() / 60) % 1440).round(1)
    hour = ordered.dt.components.hours
    out["meal_time"] = np.select([hour.between(5, 10), hour.between(11, 15), hour.between(16, 21)],
                                 ["Breakfast", "Lunch", "Dinner"], "Late Night")
    out.loc[ordered.isna(), "meal_time"] = np.nan

    out["rating"] = numeric("Delivery_person_Ratings").where(lambda s: s.between(1, 5))
    out["traffic"] = category(column("Road_traffic_density"))
    weather = column("Weatherconditions", "Weather_conditions").astype(str).str.replace("conditions", "", regex=False)
    out["weather"] = category(weather)
    out["city"] = category(column("City"))
    return out.dropna(axis=1, how="all")

def clean_data(raw, sla):
    df = raw.copy()
    log = []

    features = [c for c in df.columns if c != "order_id"]
    before = len(df)
    df = df.drop_duplicates(subset=features)
    log.append(f"Removed **{before - len(df)}** duplicate rows")

    missing_target = int(df.delivery_time_min.isna().sum())
    df = df.dropna(subset=["delivery_time_min"])
    log.append(f"Dropped **{missing_target}** rows with missing target (delivery time)")

    for col in df.columns:
        if col == "order_id":
            continue
        missing = int(df[col].isna().sum())
        if not missing:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
            method = "median"
        else:
            df[col] = df[col].fillna(df[col].mode()[0])
            method = "mode"
        log.append(f"Imputed **{missing}** missing values in `{col}` using {method}")

    q1, q3 = df.delivery_time_min.quantile([.25, .75])
    iqr = q3 - q1
    low, high = max(q1 - 3 * iqr, 1), q3 + 3 * iqr
    keep = df.delivery_time_min.between(low, high)
    log.append(f"Removed **{int((~keep).sum())}** extreme outliers outside [{low:.1f}, {high:.1f}] min (3×IQR rule)")
    df = df[keep].reset_index(drop=True)

    df["is_late"] = (df.delivery_time_min > sla).astype(int)
    log.append(f"Created feature `is_late` (delivery time > {sla:g} min)")
    return df, log


Appending to backend.py


In [ ]:
%%writefile -a backend.py
ROUTES = {}

def endpoint(name):
    def register(fn):
        ROUTES[name] = fn
        return fn
    return register

def data():
    if STATE["clean"] is None:
        raise ValueError("No data loaded yet. Go to tab 1 and run the pipeline first.")
    return STATE["clean"]

def require_columns(df, *cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Column(s) missing in dataset: {missing}")

def levels_of(df, col):
    found = sorted(df[col].astype(str).unique())
    preferred = LEVEL_ORDER.get(col, [])
    return [l for l in preferred if l in found] + [l for l in found if l not in preferred]

def group_values(df, col, level):
    return df.loc[df[col].astype(str) == level, "delivery_time_min"]

def markdown_table(df, index=True):
    d = df.reset_index() if index else df
    header = "| " + " | ".join(map(str, d.columns)) + " |\n|" + "---|" * len(d.columns) + "\n"
    rows = "\n".join(
        "| " + " | ".join(f"{v:.4g}" if isinstance(v, (float, np.floating)) else str(v) for v in row) + " |"
        for row in d.itertuples(index=False))
    return header + rows

def encode_figure(fig):
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    return base64.b64encode(buf.getvalue()).decode()

def result(text, fig=None, note=""):
    return {"text": text, "fig": encode_figure(fig) if fig is not None else None, "note": note}

def verdict(p, alpha):
    label = "❌ **Reject H₀**" if p < alpha else "✅ **Fail to reject H₀**"
    return f"{label} (p = {p:.4g}, α = {alpha})"

def cohen_label(d):
    d = abs(d)
    return "negligible" if d < .2 else "small" if d < .5 else "medium" if d < .8 else "large"


Appending to backend.py


In [ ]:
%%writefile -a backend.py
@endpoint("load")
def ep_load(p):
    path = find_dataset()
    raw = standardize(pd.read_csv(path))
    sla = float(p.get("sla", 35))
    df, log = clean_data(raw, sla)
    STATE.update(clean=df, sla=sla, model=None, formula=None)

    fig = Figure(figsize=(10, 3.6))
    before, after = fig.subplots(1, 2)
    before.hist(raw.delivery_time_min.dropna(), bins=50, color="#e76f51")
    before.set_title("Raw delivery time (before cleaning)")
    after.hist(df.delivery_time_min, bins=50, color="#2a9d8f")
    after.set_title("Cleaned delivery time")
    for ax in (before, after):
        ax.set_xlabel("minutes")

    text = (f"## ✅ Pipeline complete  ·  source: Kaggle food delivery dataset (`{path}`)\n"
            f"**Raw shape:** {raw.shape}  →  **Clean shape:** {df.shape}\n\n### Cleaning log\n"
            + "\n".join(f"- {line}" for line in log)
            + "\n\n### Preview\n" + markdown_table(df.head(8), index=False))
    return result(text, fig, f"rows {len(raw)}->{len(df)}")

@endpoint("levels")
def ep_levels(p):
    df = data()
    require_columns(df, p["col"])
    return {"levels": levels_of(df, p["col"])}

@endpoint("export")
def ep_export(p):
    return {"csv": data().to_csv(index=False)}

@endpoint("eda")
def ep_eda(p):
    df = data()
    x = df.delivery_time_min
    numeric = df.select_dtypes("number").drop(columns=["order_id", "is_late"], errors="ignore")
    summary = numeric.describe().T
    summary["skew"] = numeric.skew()
    summary["kurtosis"] = numeric.kurt()
    summary.index.name = "variable"
    shapiro = stats.shapiro(x.sample(min(len(x), 5000), random_state=1))
    dagostino = stats.normaltest(x)

    fig = Figure(figsize=(11, 8))
    ax = fig.subplots(2, 2).ravel()
    ax[0].hist(x, bins=50, density=True, color="#2a9d8f", alpha=.8)
    grid = np.linspace(x.min(), x.max(), 200)
    ax[0].plot(grid, stats.norm.pdf(grid, x.mean(), x.std()), "r--", label="Normal fit")
    ax[0].legend()
    ax[0].set_title("Distribution of delivery time")
    ax[0].set_xlabel("minutes")
    stats.probplot(x, dist="norm", plot=ax[1])
    ax[1].set_title("Q–Q plot vs Normal")
    if "traffic" in df:
        order = levels_of(df, "traffic")
        ax[2].boxplot([group_values(df, "traffic", l) for l in order])
        ax[2].set_xticks(range(1, len(order) + 1))
        ax[2].set_xticklabels(order)
        ax[2].set_title("Delivery time by traffic")
    if "distance_km" in df:
        ax[3].scatter(df.distance_km, x, s=5, alpha=.3, color="#264653")
        ax[3].set_xlabel("distance (km)")
        ax[3].set_ylabel("minutes")
        ax[3].set_title("Distance vs delivery time")

    normal = shapiro.pvalue >= .05
    reading = ("No strong evidence against normality." if normal
               else "Data deviates from normality (right-skewed, as expected for delivery times).")
    text = ("## 📊 Exploratory Data Analysis\n"
            + markdown_table(summary[["count", "mean", "std", "min", "50%", "max", "skew", "kurtosis"]])
            + "\n\n### Normality of delivery time (α = 0.05)\n"
            f"- Shapiro–Wilk: W = {shapiro.statistic:.4f}, p = {shapiro.pvalue:.4g}\n"
            f"- D'Agostino–Pearson: stat = {dagostino.statistic:.3f}, p = {dagostino.pvalue:.4g}\n"
            f"- Skewness = {stats.skew(x):.3f}, excess kurtosis = {stats.kurtosis(x):.3f}\n\n"
            f"**Interpretation:** {reading} With n = {len(x)}, the Central Limit Theorem keeps t-based "
            "inference on means reliable; non-parametric tests are reported alongside as cross-checks.")
    return result(text, fig, f"Shapiro p={shapiro.pvalue:.3g}")

@endpoint("ci")
def ep_ci(p):
    df = data()
    x = df.delivery_time_min.values
    n = len(x)
    conf = float(p.get("conf", .95))
    resamples = int(p.get("B", 2000))
    alpha = 1 - conf

    mean, sem = x.mean(), stats.sem(x)
    t_ci = stats.t.interval(conf, n - 1, loc=mean, scale=sem)

    rng = np.random.default_rng(0)
    boot_mean, boot_median = np.empty(resamples), np.empty(resamples)
    for start in range(0, resamples, 100):
        size = min(100, resamples - start)
        draws = rng.choice(x, (size, n))
        boot_mean[start:start + size] = draws.mean(1)
        boot_median[start:start + size] = np.median(draws, 1)
    bounds = [alpha / 2 * 100, (1 - alpha / 2) * 100]
    mean_ci, median_ci = np.percentile(boot_mean, bounds), np.percentile(boot_median, bounds)

    sd = x.std(ddof=1)
    sd_ci = (sd * np.sqrt((n - 1) / stats.chi2.ppf(1 - alpha / 2, n - 1)),
             sd * np.sqrt((n - 1) / stats.chi2.ppf(alpha / 2, n - 1)))
    late = int(df.is_late.sum())
    late_ci = proportion_confint(late, n, alpha=alpha, method="wilson")

    fig = Figure(figsize=(9, 4))
    ax = fig.subplots()
    ax.hist(boot_mean, bins=40, color="#8ab17d", alpha=.85)
    ax.axvline(mean_ci[0], color="r", ls="--", label="Bootstrap CI")
    ax.axvline(mean_ci[1], color="r", ls="--")
    ax.axvline(t_ci[0], color="b", ls=":", label="t-based CI")
    ax.axvline(t_ci[1], color="b", ls=":")
    ax.set_title(f"Bootstrap distribution of the mean ({resamples} resamples)")
    ax.set_xlabel("mean delivery time (min)")
    ax.legend()

    text = (f"## 🎯 {conf:.0%} Confidence Intervals  (n = {n})\n"
            f"| Parameter | Estimate | {conf:.0%} CI | Method |\n|---|---|---|---|\n"
            f"| Mean | {mean:.2f} | ({t_ci[0]:.2f}, {t_ci[1]:.2f}) | Student's t |\n"
            f"| Mean | {mean:.2f} | ({mean_ci[0]:.2f}, {mean_ci[1]:.2f}) | Bootstrap percentile |\n"
            f"| Median | {np.median(x):.2f} | ({median_ci[0]:.2f}, {median_ci[1]:.2f}) | Bootstrap percentile |\n"
            f"| Std deviation | {sd:.2f} | ({sd_ci[0]:.2f}, {sd_ci[1]:.2f}) | Chi-square |\n"
            f"| Late-delivery rate (> {STATE['sla']:g} min) | {late / n:.3%} | ({late_ci[0]:.3%}, {late_ci[1]:.3%}) | Wilson |\n\n"
            f"**Interpretation:** we are {conf:.0%} confident the true mean delivery time lies between "
            f"{t_ci[0]:.2f} and {t_ci[1]:.2f} minutes.")
    return result(text, fig, f"mean CI ({t_ci[0]:.2f},{t_ci[1]:.2f})")


Appending to backend.py


In [ ]:
%%writefile -a backend.py
@endpoint("ttest1")
def ep_ttest1(p):
    df = data()
    x = df.delivery_time_min
    mu0, alt, alpha = float(p["mu0"]), p["alt"], float(p["alpha"])
    t_stat, p_value = stats.ttest_1samp(x, mu0, alternative=alt)
    wilcoxon = stats.wilcoxon(x - mu0, alternative=alt)
    effect = (x.mean() - mu0) / x.std(ddof=1)
    h1 = {"two-sided": "μ ≠ μ₀", "greater": "μ > μ₀", "less": "μ < μ₀"}[alt]

    fig = Figure(figsize=(9, 4))
    ax = fig.subplots()
    ax.hist(x, bins=50, color="#a8dadc")
    ax.axvline(mu0, color="r", ls="--", label=f"H₀: μ₀ = {mu0:g}")
    ax.axvline(x.mean(), color="k", label=f"Sample mean = {x.mean():.2f}")
    ax.legend()
    ax.set_title("One-sample test: sample vs hypothesised mean")
    ax.set_xlabel("minutes")

    text = (f"## 🧪 One-Sample t-test\n**H₀:** μ = {mu0:g} min  |  **H₁:** {h1}\n\n"
            "| Test | Statistic | p-value |\n|---|---|---|\n"
            f"| One-sample t (df = {len(x) - 1}) | {t_stat:.3f} | {p_value:.4g} |\n"
            f"| Wilcoxon signed-rank (non-parametric) | {wilcoxon.statistic:.1f} | {wilcoxon.pvalue:.4g} |\n\n"
            f"- Sample mean = {x.mean():.2f}, SD = {x.std():.2f}\n"
            f"- Cohen's d = {effect:.3f} ({cohen_label(effect)} effect)\n\n"
            f"**Decision (t-test):** {verdict(p_value, alpha)}")
    return result(text, fig, f"t={t_stat:.2f}, p={p_value:.3g}")

@endpoint("ttest2")
def ep_ttest2(p):
    df = data()
    col, a, b, alt, alpha = p["col"], p["a"], p["b"], p["alt"], float(p["alpha"])
    require_columns(df, col)
    xa, xb = group_values(df, col, a), group_values(df, col, b)
    if a == b or len(xa) < 2 or len(xb) < 2:
        raise ValueError("Pick two different levels with enough observations.")

    levene = stats.levene(xa, xb)
    welch = stats.ttest_ind(xa, xb, equal_var=False, alternative=alt)
    mann = stats.mannwhitneyu(xa, xb, alternative=alt)
    na, nb, va, vb = len(xa), len(xb), xa.var(ddof=1), xb.var(ddof=1)
    diff = xa.mean() - xb.mean()
    se = np.sqrt(va / na + vb / nb)
    dof = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    margin = stats.t.ppf(1 - alpha / 2, dof) * se
    pooled = np.sqrt(((na - 1) * va + (nb - 1) * vb) / (na + nb - 2))
    effect = diff / pooled
    power = TTestIndPower().power(effect_size=abs(effect), nobs1=na, ratio=nb / na, alpha=alpha)
    h1 = {"two-sided": "μ_A ≠ μ_B", "greater": "μ_A > μ_B", "less": "μ_A < μ_B"}[alt]

    fig = Figure(figsize=(8, 4.5))
    ax = fig.subplots()
    ax.violinplot([xa, xb], showmeans=True)
    ax.set_xticks([1, 2])
    ax.set_xticklabels([f"{a} (n={na})", f"{b} (n={nb})"])
    ax.set_ylabel("delivery time (min)")
    ax.set_title(f"Delivery time: {a} vs {b}")

    text = (f"## ⚖️ Two-Sample Test on `{col}`: **{a}** (A) vs **{b}** (B)\n"
            f"**H₀:** μ_A = μ_B  |  **H₁:** {h1}\n\n"
            "| Group | n | Mean | SD |\n|---|---|---|---|\n"
            f"| {a} | {na} | {xa.mean():.2f} | {xa.std():.2f} |\n"
            f"| {b} | {nb} | {xb.mean():.2f} | {xb.std():.2f} |\n\n"
            "| Test | Statistic | p-value |\n|---|---|---|\n"
            f"| Levene (equal variances) | {levene.statistic:.3f} | {levene.pvalue:.4g} |\n"
            f"| Welch t-test (df ≈ {dof:.0f}) | {welch.statistic:.3f} | {welch.pvalue:.4g} |\n"
            f"| Mann–Whitney U | {mann.statistic:.1f} | {mann.pvalue:.4g} |\n\n"
            f"- Mean difference (A−B) = {diff:.2f} min, {1 - alpha:.0%} CI ({diff - margin:.2f}, {diff + margin:.2f})\n"
            f"- Cohen's d = {effect:.3f} ({cohen_label(effect)}); achieved power ≈ {power:.3f}\n\n"
            f"**Decision (Welch t):** {verdict(welch.pvalue, alpha)}")
    return result(text, fig, f"{col}: {a} vs {b}, p={welch.pvalue:.3g}")

@endpoint("anova")
def ep_anova(p):
    df = data()
    col, alpha = p["col"], float(p["alpha"])
    require_columns(df, col)
    levels = [l for l in levels_of(df, col) if (df[col].astype(str) == l).sum() > 1]
    if len(levels) < 3:
        raise ValueError("ANOVA needs a factor with at least 3 levels (use the two-sample test for 2).")

    groups = [group_values(df, col, l) for l in levels]
    f_stat, p_value = stats.f_oneway(*groups)
    h_stat, p_kruskal = stats.kruskal(*groups)
    levene = stats.levene(*groups)
    x = df.delivery_time_min
    grand = x.mean()
    eta_sq = sum(len(g) * (g.mean() - grand) ** 2 for g in groups) / ((x - grand) ** 2).sum()
    tukey = pairwise_tukeyhsd(x, df[col].astype(str), alpha=alpha)
    tukey_table = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])

    means = [g.mean() for g in groups]
    errors = [stats.t.ppf(1 - alpha / 2, len(g) - 1) * stats.sem(g) for g in groups]
    fig = Figure(figsize=(8, 4.5))
    ax = fig.subplots()
    ax.errorbar(range(len(levels)), means, yerr=errors, fmt="o", capsize=6, color="#e76f51", markersize=8)
    ax.set_xticks(range(len(levels)))
    ax.set_xticklabels(levels)
    ax.set_ylabel("mean delivery time (min)")
    ax.set_title(f"Group means with {1 - alpha:.0%} CI  ({col})")

    text = (f"## 🧮 One-Way ANOVA on `{col}`\n**H₀:** all group means are equal  |  **H₁:** at least one differs\n\n"
            "| Test | Statistic | p-value |\n|---|---|---|\n"
            f"| Levene (homogeneity) | {levene.statistic:.3f} | {levene.pvalue:.4g} |\n"
            f"| ANOVA F | {f_stat:.3f} | {p_value:.4g} |\n"
            f"| Kruskal–Wallis H | {h_stat:.3f} | {p_kruskal:.4g} |\n\n"
            f"- Effect size η² = {eta_sq:.3f}\n\n**Decision (ANOVA):** {verdict(p_value, alpha)}\n\n"
            "### Tukey HSD post-hoc\n" + markdown_table(tukey_table, index=False))
    return result(text, fig, f"{col}: F={f_stat:.2f}, p={p_value:.3g}")

@endpoint("chi2")
def ep_chi2(p):
    df = data()
    col, alpha = p["col"], float(p["alpha"])
    require_columns(df, col)
    table = pd.crosstab(df[col].astype(str), df.is_late)
    if table.shape[1] < 2:
        raise ValueError("Need both late and on-time orders. Adjust the late threshold in tab 1.")
    table.columns = ["On time", "Late"]
    chi, p_value, dof, expected = stats.chi2_contingency(table)
    cramers_v = np.sqrt(chi / (table.values.sum() * (min(table.shape) - 1)))
    late_rate = table["Late"] / table.sum(axis=1) * 100

    fig = Figure(figsize=(8, 4.5))
    ax = fig.subplots()
    ax.bar(late_rate.index, late_rate.values, color="#f4a261")
    ax.set_ylabel(f"% late (> {STATE['sla']:g} min)")
    ax.set_title(f"Late-delivery rate by {col}")

    assumption = "assumption OK" if expected.min() >= 5 else "assumption violated"
    text = (f"## 🔗 Chi-Square Test of Independence: `{col}` × Late delivery\n"
            f"**H₀:** lateness is independent of `{col}`  |  **H₁:** they are associated\n\n"
            + markdown_table(table)
            + f"\n\n- χ² = {chi:.3f}, df = {dof}, p = {p_value:.4g}\n"
            f"- Cramér's V = {cramers_v:.3f}\n"
            f"- Min expected count = {expected.min():.1f} ({assumption})\n\n"
            f"**Decision:** {verdict(p_value, alpha)}")
    return result(text, fig, f"{col}: chi2={chi:.2f}, p={p_value:.3g}")

@endpoint("corr")
def ep_corr(p):
    df = data()
    var, alpha = p["var"], float(p.get("alpha", .05))
    require_columns(df, var)
    x, y = df[var], df.delivery_time_min
    n = len(x)
    pearson_r, pearson_p = stats.pearsonr(x, y)
    rho, rho_p = stats.spearmanr(x, y)
    tau, tau_p = stats.kendalltau(x, y)
    z, z_crit = np.arctanh(pearson_r), stats.norm.ppf(1 - alpha / 2)
    low = np.tanh(z - z_crit / np.sqrt(n - 3))
    high = np.tanh(z + z_crit / np.sqrt(n - 3))

    fig = Figure(figsize=(8, 4.5))
    ax = fig.subplots()
    ax.scatter(x, y, s=6, alpha=.3, color="#264653")
    slope = np.polyfit(x, y, 1)
    grid = np.linspace(x.min(), x.max(), 100)
    ax.plot(grid, np.polyval(slope, grid), "r-")
    ax.set_xlabel(var)
    ax.set_ylabel("delivery time (min)")
    ax.set_title(f"{var} vs delivery time (r = {pearson_r:.3f})")

    text = (f"## 📈 Correlation: `{var}` vs delivery time\n**H₀:** ρ = 0\n\n"
            "| Measure | Coefficient | p-value |\n|---|---|---|\n"
            f"| Pearson r | {pearson_r:.4f} | {pearson_p:.4g} |\n"
            f"| Spearman ρ | {rho:.4f} | {rho_p:.4g} |\n"
            f"| Kendall τ | {tau:.4f} | {tau_p:.4g} |\n\n"
            f"- Pearson {1 - alpha:.0%} CI (Fisher z): ({low:.4f}, {high:.4f})\n"
            f"- R² (simple linear) = {pearson_r ** 2:.3f}\n\n"
            f"**Decision:** {verdict(pearson_p, alpha)}")
    return result(text, fig, f"{var}: r={pearson_r:.3f}")


Appending to backend.py


In [ ]:
%%writefile -a backend.py
@endpoint("regression")
def ep_regression(p):
    df = data()
    alpha, robust = float(p.get("alpha", .05)), bool(p.get("robust", True))
    terms = ([c for c in ["distance_km", "prep_time_min", "rating"] if c in df]
             + [f"C({c})" for c in ["traffic", "weather", "meal_time"] if c in df])
    if not terms:
        raise ValueError("No predictor columns found in dataset.")
    formula = "delivery_time_min ~ " + " + ".join(terms)

    plain = smf.ols(formula, df).fit()
    model = smf.ols(formula, df).fit(cov_type="HC3") if robust else plain
    STATE.update(model=plain, formula=formula)

    bounds = model.conf_int(alpha=alpha)
    coefficients = pd.DataFrame({"coef": model.params, "std err": model.bse, "t/z": model.tvalues,
                                 "p": model.pvalues, "CI low": bounds[0], "CI high": bounds[1]})
    coefficients.index.name = "term"

    residuals = plain.resid
    shapiro = stats.shapiro(residuals.sample(min(len(residuals), 5000), random_state=1))
    breusch_pagan = het_breuschpagan(plain.resid, plain.model.exog)[1]

    order = np.random.default_rng(1).permutation(len(df))
    cut = int(.8 * len(df))
    train, test = df.iloc[order[:cut]], df.iloc[order[cut:]]
    errors = test.delivery_time_min - smf.ols(formula, train).fit().predict(test)
    rmse, mae = np.sqrt((errors ** 2).mean()), errors.abs().mean()
    test_r2 = 1 - (errors ** 2).sum() / ((test.delivery_time_min - test.delivery_time_min.mean()) ** 2).sum()

    fig = Figure(figsize=(11, 4.5))
    left, right = fig.subplots(1, 2)
    left.scatter(plain.fittedvalues, residuals, s=5, alpha=.3, color="#264653")
    left.axhline(0, color="r")
    left.set_xlabel("fitted")
    left.set_ylabel("residual")
    left.set_title("Residuals vs fitted")
    stats.probplot(residuals, dist="norm", plot=right)
    right.set_title("Residual Q–Q plot")

    se_label = ", HC3 robust SE" if robust else ""
    hetero = ("→ heteroscedastic, so robust SE are appropriate" if breusch_pagan < .05 else "→ homoscedastic")
    text = (f"## 🤖 Multiple Linear Regression (OLS{se_label})\n`{formula}`\n\n"
            f"- R² = {plain.rsquared:.4f}, Adj. R² = {plain.rsquared_adj:.4f}\n"
            f"- Overall F-test: F = {plain.fvalue:.1f}, p = {plain.f_pvalue:.3g}\n\n"
            "### Coefficients\n" + markdown_table(coefficients)
            + "\n\n### Diagnostics\n"
            f"- Residual Shapiro–Wilk p = {shapiro.pvalue:.4g}\n"
            f"- Breusch–Pagan (heteroscedasticity) p = {breusch_pagan:.4g} {hetero}\n\n"
            "### 80/20 hold-out validation\n"
            f"- RMSE = {rmse:.2f} min, MAE = {mae:.2f} min, test R² = {test_r2:.3f}")
    return result(text, fig, f"R2={plain.rsquared:.3f}")

@endpoint("predict")
def ep_predict(p):
    model = STATE["model"]
    if model is None:
        raise ValueError("Train the regression model first (Regression tab → Fit model).")
    row = pd.DataFrame([{"distance_km": float(p["distance"]), "prep_time_min": float(p["prep"]),
                         "rating": float(p["rating"]), "traffic": p["traffic"],
                         "weather": p["weather"], "meal_time": p["meal"]}])
    frame = model.get_prediction(row).summary_frame(alpha=.05).iloc[0]
    sigma = np.sqrt(model.scale)
    p_late = 1 - stats.norm.cdf((STATE["sla"] - frame["mean"]) / sigma)
    text = (f"## 🛵 Predicted delivery time\n# **{frame['mean']:.1f} minutes**\n\n"
            f"- 95% CI for the *mean* delivery time of such orders: ({frame['mean_ci_lower']:.1f}, {frame['mean_ci_upper']:.1f})\n"
            f"- 95% **prediction interval** for *this* order: ({frame['obs_ci_lower']:.1f}, {frame['obs_ci_upper']:.1f})\n"
            f"- Estimated P(late > {STATE['sla']:g} min) ≈ {p_late:.1%} (normal-residual approximation)")
    return result(text, None, f"pred {frame['mean']:.1f} min")

@endpoint("history")
def ep_history(p):
    rows = db_exec("SELECT ts, endpoint, params, result FROM analyses ORDER BY id DESC LIMIT 25")
    if not rows:
        return result("No analyses logged yet.")
    history = pd.DataFrame(rows, columns=["time (UTC)", "analysis", "parameters", "result"])
    history = history.apply(lambda s: s.astype(str).str.replace("|", "/", regex=False).str[:90])
    return result("## 🗂️ Analysis history (SQLite)\n" + markdown_table(history, index=False))


Appending to backend.py


In [ ]:
%%writefile -a backend.py
app = Flask(__name__)

@app.post("/api/<name>")
def api(name):
    handler = ROUTES.get(name)
    if handler is None:
        return jsonify({"text": f"⚠️ **Error:** unknown endpoint `{name}`", "fig": None, "note": "error"}), 404
    params = request.get_json(force=True, silent=True) or {}
    try:
        reply = handler(params)
        if name not in UNLOGGED:
            db_log(name, params, reply.get("note", ""))
        return jsonify(reply)
    except Exception as e:
        return jsonify({"text": f"⚠️ **Error:** {e}", "fig": None, "note": "error"})

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5055, threaded=True)


Appending to backend.py


In [ ]:
%%writefile app.py
import base64
import requests
import streamlit as st

API = "http://127.0.0.1:5055/api/"
CATEGORICAL = ["traffic", "weather", "meal_time", "city"]
NUMERIC = ["distance_km", "prep_time_min", "rating"]
ALTERNATIVES = ["two-sided", "greater", "less"]
PREDICT_DEFAULTS = {"traffic": ("Medium",), "weather": ("Sunny", "Clear"), "meal_time": ("Dinner",)}
GROUP_DEFAULTS = {"weather": ("Stormy", "Sunny"), "traffic": ("High", "Low"),
                  "meal_time": ("Dinner", "Breakfast"), "city": ("Metropolitian", "Urban")}
TAB_TITLES = ["1 · Data & Pipeline", "2 · EDA & Normality", "3 · Confidence Intervals",
              "4 · One-Sample Test", "5 · Two-Sample Test", "6 · ANOVA", "7 · Chi-Square",
              "8 · Correlation", "9 · Regression & Prediction", "10 · History (DB)"]

st.set_page_config(page_title="Food Delivery Time Statistical Analyzer", page_icon="🍔", layout="wide")


Writing app.py


In [ ]:
%%writefile -a app.py
def api(endpoint, **params):
    try:
        return requests.post(API + endpoint, json=params, timeout=900).json()
    except Exception as e:
        return {"text": f"⚠️ **Backend not reachable:** {e}", "fig": None, "note": "error"}

def show(key):
    reply = st.session_state.get(key)
    if not reply:
        return
    st.markdown(reply.get("text", ""))
    if reply.get("fig"):
        st.image(base64.b64decode(reply["fig"]))

def run(key, label, endpoint, **params):
    if st.button(label, type="primary", key=f"btn_{key}"):
        with st.spinner("Running…"):
            st.session_state[key] = api(endpoint, **params)
    show(key)

def require_data():
    if "levels" not in st.session_state:
        st.info("Run the pipeline in tab 1 first.")
        return False
    return True

def level_options(col):
    return st.session_state.get("levels", {}).get(col, [])

def categorical_columns():
    return [c for c in CATEGORICAL if level_options(c)]

def default_index(options, preferred):
    for value in preferred:
        if value in options:
            return options.index(value)
    return 0

def alpha_slider(key):
    return round(st.slider("Significance α", 0.01, 0.10, 0.05, 0.01, key=key), 2)


Appending to app.py


In [ ]:
%%writefile -a app.py
st.title("🍔 Food Delivery Time Statistical Analyzer")
st.caption("Statistical Inference mini-project · end-to-end pipeline: acquire → clean → EDA → "
           "confidence intervals → hypothesis tests → ANOVA → χ² → correlation → regression → prediction")

tabs = st.tabs(TAB_TITLES)

with tabs[0]:
    sla = st.number_input("Late threshold (min)", min_value=1.0, value=35.0, step=1.0)
    if st.button("▶ Run pipeline", type="primary", key="btn_load"):
        with st.spinner("Loading and cleaning the Kaggle dataset…"):
            reply = api("load", sla=float(sla))
            st.session_state["load"] = reply
            if reply.get("note") == "error":
                st.session_state.pop("levels", None)
                st.session_state.pop("csv", None)
            else:
                st.session_state["levels"] = {c: api("levels", col=c).get("levels", []) for c in CATEGORICAL}
                st.session_state["csv"] = api("export").get("csv")
    if st.session_state.get("csv"):
        st.download_button("⬇ Download cleaned CSV", st.session_state["csv"],
                           file_name="clean_food_delivery.csv", mime="text/csv")
    show("load")

with tabs[1]:
    if require_data():
        run("eda", "▶ Analyze", "eda")

with tabs[2]:
    if require_data():
        left, right = st.columns(2)
        confidence = left.slider("Confidence level", 0.80, 0.99, 0.95, 0.01)
        resamples = right.slider("Bootstrap resamples", 500, 5000, 2000, 500)
        run("ci", "▶ Compute", "ci", conf=round(confidence, 2), B=int(resamples))


Appending to app.py


In [ ]:
%%writefile -a app.py
with tabs[3]:
    if require_data():
        left, middle, right = st.columns(3)
        mu0 = left.number_input("Promised / hypothesised mean μ₀ (min)", value=30.0)
        alternative = middle.radio("Alternative", ALTERNATIVES, key="alt_one", horizontal=True)
        with right:
            alpha = alpha_slider("alpha_one")
        run("ttest1", "▶ Run test", "ttest1", mu0=float(mu0), alt=alternative, alpha=alpha)

with tabs[4]:
    if require_data():
        col = st.selectbox("Grouping column", categorical_columns(), key="two_col")
        options = level_options(col)
        want_a, want_b = GROUP_DEFAULTS.get(col, (None, None))
        index_a = default_index(options, (want_a,))
        index_b = default_index(options, (want_b,))
        if index_b == index_a:
            index_b = (index_a + 1) % len(options)
        left, right = st.columns(2)
        group_a = left.selectbox("Group A", options, index=index_a, key=f"group_a_{col}_{len(options)}")
        group_b = right.selectbox("Group B", options, index=index_b, key=f"group_b_{col}_{len(options)}")
        left, right = st.columns(2)
        alternative = left.radio("Alternative", ALTERNATIVES, key="alt_two", horizontal=True)
        with right:
            alpha = alpha_slider("alpha_two")
        run("ttest2", "▶ Run test", "ttest2", col=col, a=group_a, b=group_b, alt=alternative, alpha=alpha)

with tabs[5]:
    if require_data():
        left, right = st.columns(2)
        factor = left.selectbox("Factor", categorical_columns(), key="anova_col")
        with right:
            alpha = alpha_slider("alpha_anova")
        run("anova", "▶ Run ANOVA + Tukey", "anova", col=factor, alpha=alpha)

with tabs[6]:
    if require_data():
        left, right = st.columns(2)
        factor = left.selectbox("Categorical factor", categorical_columns(), key="chi_col")
        with right:
            alpha = alpha_slider("alpha_chi")
        run("chi2", "▶ Run χ² test", "chi2", col=factor, alpha=alpha)

with tabs[7]:
    if require_data():
        variable = st.selectbox("Numeric variable", NUMERIC, key="corr_var")
        run("corr", "▶ Compute", "corr", var=variable)


Appending to app.py


In [ ]:
%%writefile -a app.py
with tabs[8]:
    if require_data():
        left, right = st.columns(2)
        robust = left.checkbox("Robust (HC3) standard errors", value=True)
        with right:
            alpha = alpha_slider("alpha_reg")
        run("regression", "▶ Fit model", "regression", robust=bool(robust), alpha=alpha)

        st.markdown("---\n### Predict a delivery time")
        left, middle, right = st.columns(3)
        distance = left.slider("Distance (km)", 0.5, 25.0, 5.0, 0.5)
        prep = middle.slider("Prep time (min)", 0.0, 30.0, 10.0, 0.5)
        rating = right.slider("Rating", 1.0, 5.0, 4.5, 0.1)

        left, middle, right = st.columns(3)
        traffic_options = level_options("traffic") or ["n/a"]
        weather_options = level_options("weather") or ["n/a"]
        meal_options = level_options("meal_time") or ["n/a"]
        traffic = left.selectbox("Traffic", traffic_options,
                                 index=default_index(traffic_options, PREDICT_DEFAULTS["traffic"]),
                                 key="pred_traffic_" + "|".join(traffic_options))
        weather = middle.selectbox("Weather", weather_options,
                                   index=default_index(weather_options, PREDICT_DEFAULTS["weather"]),
                                   key="pred_weather_" + "|".join(weather_options))
        meal = right.selectbox("Meal time", meal_options,
                               index=default_index(meal_options, PREDICT_DEFAULTS["meal_time"]),
                               key="pred_meal_" + "|".join(meal_options))
        run("predict", "🔮 Predict", "predict", distance=float(distance), prep=float(prep),
            rating=round(rating, 1), traffic=traffic, weather=weather, meal=meal)

with tabs[9]:
    run("history", "🔄 Refresh", "history")

st.markdown("---")
st.caption("**Architecture:** Kaggle dataset (extracted train.csv) → Streamlit frontend → HTTP/JSON → "
           "Flask REST backend (scipy · statsmodels · pandas) → SQLite log")


Appending to app.py


In [ ]:
import os, re, sys, time, subprocess, urllib.request, requests

os.chdir("/content")

for pat in ("backend.py", "streamlit run", "cloudflared"):
    subprocess.run(["pkill", "-f", pat], check=False)
time.sleep(1)

def wait_for(check, seconds, what):
    for _ in range(int(seconds * 2)):
        try:
            if check():
                return True
        except Exception:
            pass
        time.sleep(.5)
    print(f"{what} did not start in time, check the matching .log file in /content")
    return False

subprocess.Popen([sys.executable, "backend.py"], stdout=open("backend.log", "w"), stderr=subprocess.STDOUT)
if wait_for(lambda: requests.post("http://127.0.0.1:5055/api/history", json={}, timeout=2).ok, 40, "Backend"):
    print("Backend running at http://127.0.0.1:5055/api/")


Backend running at http://127.0.0.1:5055/api/


In [ ]:
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py",
                  "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
                  "--browser.gatherUsageStats", "false"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
if wait_for(lambda: requests.get("http://127.0.0.1:8501/_stcore/health", timeout=2).text.strip() == "ok", 60, "Streamlit"):
    print("Streamlit running at http://127.0.0.1:8501")


Streamlit running at http://127.0.0.1:8501


In [ ]:
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)

subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
                 stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(80):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break
    time.sleep(.5)
print("Open your app here:", url or "tunnel failed, see tunnel.log")


Open your app here: https://anywhere-solid-returns-dash.trycloudflare.com
